# EDA — `cuidados_plantas.csv`

Estudio exploratorio del dataset de cuidados de plantas usado por la API Plant Care (búsqueda semántica / RAG).

**Columnas:** `id`, `species_id`, `name`, `scientific_name`, `sunlight`, `watering`, `pruning`

**Objetivos:**
- Conocer volumen, tipos y calidad de los datos
- Analizar longitud y contenido de los textos de cuidado
- Detectar duplicados y posibles errores (p. ej. columnas permutadas)
- Limpieza ligera **solo si** el análisis lo justifica

In [ ]:
!pip install -q pandas matplotlib seaborn

## Cargar el CSV

En **Colab**, elige una opción:

1. **Clonar el repo** (si es público):
   ```python
   !git clone https://github.com/TU_USUARIO/Gaia.git
   ```
2. **Subir** `cuidados_plantas.csv` al directorio de Colab.
3. **Google Drive** — monta Drive y ajusta `CSV_PATH` abajo.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

CSV_PATH = 'cuidados_plantas.csv'

df_raw = pd.read_csv(CSV_PATH)
df_raw.head()

## 1. Vista general

In [ ]:
print("Filas:", len(df_raw), "| Columnas:", len(df_raw.columns))
print("\nTipos:")
display(df_raw.dtypes.to_frame("dtype"))

print("\nValores nulos:")
display(df_raw.isnull().sum().to_frame("nulos"))

print("\nMemoria (aprox.):", f"{df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df_raw.describe(include="all").T

In [ ]:
TEXT_COLS = ["sunlight", "watering", "pruning"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, TEXT_COLS):
    lengths = df_raw[col].astype(str).str.len()
    ax.hist(lengths, bins=40, color="#2e7d32", edgecolor="white")
    ax.set_title(f"Longitud de texto — {col}")
    ax.set_xlabel("caracteres")
    ax.set_ylabel("frecuencia")
plt.tight_layout()
plt.show()

pd.DataFrame({
    c: df_raw[c].astype(str).str.len().describe() for c in TEXT_COLS
}).T

## 2. Identificadores y duplicados

In [ ]:
dup_id = df_raw["id"].duplicated().sum()
dup_name = df_raw["name"].duplicated().sum()
dup_species = df_raw["species_id"].duplicated().sum()
dup_sci = df_raw["scientific_name"].duplicated().sum()

print(f"id duplicados: {dup_id}")
print(f"name duplicados: {dup_name}")
print(f"species_id duplicados: {dup_species}")
print(f"scientific_name duplicados: {dup_sci}")

if dup_name:
    print("\nNombres más repetidos:")
    display(df_raw["name"].value_counts().head(10))

n_species = df_raw["name"].nunique()
print(f"\nEspecies distintas (name): {n_species} de {len(df_raw)} filas")

## 3. Nombre científico

El campo suele venir como JSON en string, p. ej. `["Malus 'Ambrosia'"]`.

In [ ]:
import json
import re


def parse_scientific_name(value) -> str:
    if pd.isna(value):
        return ""
    s = str(value).strip()
    try:
        parsed = json.loads(s.replace('""', '"'))
        if isinstance(parsed, list) and parsed:
            return str(parsed[0]).strip()
        if isinstance(parsed, str):
            return parsed.strip()
    except json.JSONDecodeError:
        pass
    m = re.search(r'\["([^"]+)"\]', s)
    return m.group(1).strip() if m else s


df_raw["scientific_name_parsed"] = df_raw["scientific_name"].map(parse_scientific_name)
df_raw[["name", "scientific_name", "scientific_name_parsed"]].head(8)

## 4. Calidad de textos — columnas permutadas

En algunas filas del CSV original, `sunlight`, `watering` y `pruning` parecen intercambiadas
(p. ej. texto de poda en la columna de luz). Usamos palabras clave por tema para estimar cuántas filas podrían estar mal asignadas.

In [ ]:
KEYWORDS = {
    "sunlight": r"\b(sunlight|sun |hours of sun|full sun|shade|direct light)\b",
    "watering": r"\b(water|watering|inch(es)? of water|soil moist|irrigation)\b",
    "pruning": r"\b(prun(e|ing)|trim|shears|dead branch|canopy)\b",
}


def score_column(text: str, topic: str) -> int:
    if pd.isna(text):
        return 0
    return len(re.findall(KEYWORDS[topic], str(text), flags=re.I))


scores = pd.DataFrame(index=df_raw.index)
for col in TEXT_COLS:
    for topic in KEYWORDS:
        scores[f"{col}_as_{topic}"] = df_raw[col].map(lambda t, tp=topic: score_column(t, tp))


def best_topic_for_row(row_idx: int) -> dict:
    row = scores.loc[row_idx]
    out = {}
    for col in TEXT_COLS:
        topic_scores = {t: row[f"{col}_as_{t}"] for t in KEYWORDS}
        out[col] = max(topic_scores, key=topic_scores.get)
    return out


mismatches = []
for idx in df_raw.index:
    assigned = best_topic_for_row(idx)
    wrong = [c for c in TEXT_COLS if assigned[c] != c]
    if wrong:
        mismatches.append({"index": idx, "id": df_raw.at[idx, "id"], "name": df_raw.at[idx, "name"], "cols_sospechosas": wrong})

print(f"Filas con posible permutación de columnas (heurística): {len(mismatches)} / {len(df_raw)}")
if mismatches:
  display(pd.DataFrame(mismatches).head(15))

In [ ]:
# Ejemplo manual: mandarin orange (id=5) — revisar si las columnas coinciden con el tema esperado
sample = df_raw[df_raw["id"] == 5]
if not sample.empty:
    for col in TEXT_COLS:
        print(f"--- {col} ---")
        print(str(sample.iloc[0][col])[:220], "...\n")

## 5. Limpieza ligera (opcional)

El dataset está en general **bien estructurado**: sin nulos masivos, textos largos pero coherentes.

Solo aplicamos:
- recorte de espacios en strings
- columna auxiliar `scientific_name_parsed`
- reordenación automática de `sunlight` / `watering` / `pruning` **solo** en filas donde la heurística detecta permutación clara

No se eliminan filas ni se reescribe texto.

In [ ]:
def reorder_care_columns(row: pd.Series) -> pd.Series:
    texts = {c: row[c] for c in TEXT_COLS}
    scored = []
    for col, text in texts.items():
        topic_scores = {t: score_column(text, t) for t in KEYWORDS}
        best = max(topic_scores, key=topic_scores.get)
        scored.append((col, best, text, max(topic_scores.values())))
    # Solo reordenar si hay desajuste y puntuación mínima
    if all(s[1] == s[0] for s in scored):
        return row
    if max(s[3] for s in scored) < 2:
        return row

    by_topic = {}
    for col, best, text, _ in scored:
        if best not in by_topic:
            by_topic[best] = text
    row = row.copy()
    for topic in KEYWORDS:
        if topic in by_topic:
            row[topic] = by_topic[topic]
    return row


df = df_raw.copy()
for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].map(lambda x: x.strip() if isinstance(x, str) else x)

df["scientific_name_parsed"] = df["scientific_name"].map(parse_scientific_name)

before = df[TEXT_COLS].copy()
df = df.apply(reorder_care_columns, axis=1)
changed = (before != df[TEXT_COLS]).any(axis=1).sum()
print(f"Filas con columnas de cuidado reordenadas: {changed}")

df.head(3)

In [ ]:
# Guardar versión limpia — no sobrescribe el CSV del repo por defecto
SAVE_CLEAN = False
OUT_PATH = Path("cuidados_plantas_limpio.csv")

if SAVE_CLEAN:
    export_cols = [
        "id", "species_id", "name", "scientific_name",
        "scientific_name_parsed", "sunlight", "watering", "pruning",
    ]
    df[export_cols].to_csv(OUT_PATH, index=False)
    print("Guardado:", OUT_PATH.resolve())
else:
    print("SAVE_CLEAN=False — no se escribe fichero. Cambia a True para exportar.")

## 6. Conclusiones

Rellena tras ejecutar todas las celdas:

- **Tamaño y completitud:** …
- **Duplicados en `name`:** esperables (variedades / cultivares distintos con mismo nombre común).
- **Longitud de textos:** adecuada para embeddings; revisar outliers muy largos.
- **Permutación de columnas:** si la heurística marca pocas filas, el impacto en RAG es limitado; si marca muchas, valorar usar `df` limpio o corregir en origen.
- **Recomendación:** para producción, el CSV original suele bastar; la limpieza automática es conservadora y opcional.